# SentenceTransformer Demo: Text Embeddings, Similarity, and a Retrieval Chatbot

This notebook demonstrates how to use `SentenceTransformer` to:

- Convert a sentence into a numerical vector.
- Compare the semantic similarity of sentences.
- Compare the semantic similarity of two documents.
- Build a simple retrieval-based chatbot from `(question, answer)` pairs.

> First run note: loading a SentenceTransformer model can take a few minutes if the model must be downloaded or initialized for the first time.

## 0. Setup

We will use the compact model `sentence-transformers/all-MiniLM-L6-v2`. It produces useful sentence embeddings while staying small enough for classroom demos.

In [1]:
%pip install sentence-transformers

  Using cached sentence_transformers-5.5.1-py3-none-any.whl.metadata (18 kB)
  Using cached transformers-5.11.0-py3-none-any.whl.metadata (33 kB)
  Using cached huggingface_hub-1.19.0-py3-none-any.whl.metadata (14 kB)
  Using cached tqdm-4.68.2-py3-none-any.whl.metadata (58 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer-0.26.7-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.2 kB)
  Using cached click-8.4.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached hf_xet-1.5.1-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cach

In [2]:
import re

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Tải mô hình SentenceTransformer; lần chạy đầu tiên có thể mất thời gian nếu cần tải model.
model = SentenceTransformer(MODEL_NAME)

print("Model loaded:", MODEL_NAME)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2267.92it/s]


Model loaded: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


/tmp/ipykernel_27510/570177198.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


## 1. Convert a Sentence into a Vector

A sentence embedding is a dense vector that represents the meaning of a sentence. Similar meanings should have similar vectors.

In [3]:
def embed_texts(texts, embedding_model, normalize=True):
    """Convert text strings into embedding vectors.

    Parameters:
        texts: A list of text strings to encode.
        embedding_model: A SentenceTransformer model used to create embeddings.
        normalize: If True, normalize vectors so dot product equals cosine similarity.

    Returns:
        A NumPy array containing one embedding vector for each input text.
    """
    # Chuyển danh sách câu thành vector; normalize_embeddings giúp so sánh cosine dễ hơn.
    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=normalize,
    )
    return embeddings


sentence = "Artificial intelligence helps computers understand human language."

# Mỗi câu được biểu diễn bằng một vector số thực nhiều chiều.
sentence_vector = embed_texts([sentence], model)[0]

print("Sentence:")
print(sentence)
print("\nVector shape:", sentence_vector.shape)
print("\nFirst 20 values:")
print(np.round(sentence_vector[:20], 4))
print("\nFull vector:")
print(sentence_vector)

Sentence:
Artificial intelligence helps computers understand human language.

Vector shape: (384,)

First 20 values:
[ 0.0214  0.0014  0.061  -0.0289 -0.0286  0.0058  0.0756  0.0087  0.0199
  0.0021  0.0016  0.0242  0.0049  0.0081  0.0571  0.0198 -0.0422  0.0318
 -0.1316 -0.1285]

Full vector:
[ 2.14219075e-02  1.39585952e-03  6.09815307e-02 -2.89216246e-02
 -2.86351386e-02  5.81759727e-03  7.56151453e-02  8.65025539e-03
  1.99439786e-02  2.09069648e-03  1.56616676e-03  2.42322776e-02
  4.90960805e-03  8.08957778e-03  5.71212620e-02  1.97828729e-02
 -4.22418229e-02  3.17515768e-02 -1.31571621e-01 -1.28459871e-01
  5.36294729e-02  1.52904801e-02 -2.69361325e-02 -2.87848245e-02
 -1.53503986e-02  8.30760747e-02 -2.20952444e-02 -4.89072986e-02
  2.34065875e-02 -6.67861756e-03  1.43558858e-02  5.23592811e-03
  1.12756893e-01  5.57112508e-02 -9.30926576e-02  6.68581277e-02
 -3.46404761e-02  1.40544930e-02  5.76246083e-02 -1.06379259e-02
 -5.90910502e-02 -5.90763316e-02  6.95611686e-02  2.681

## 2. Compare Sentence Similarity

Cosine similarity measures the angle between two vectors:

- Close to `1`: very similar meaning.
- Around `0`: weak or no semantic relationship.
- Below `0`: opposite or strongly different direction in the embedding space.

In [4]:
def cosine_similarity_matrix(embeddings):
    """Compute a pairwise cosine similarity matrix.

    Parameters:
        embeddings: A 2D NumPy array of normalized embedding vectors.

    Returns:
        A square matrix where entry (i, j) is the similarity of item i and item j.
    """
    # Vì vector đã được chuẩn hóa, tích vô hướng chính là cosine similarity.
    return embeddings @ embeddings.T


def make_similarity_table(labels, similarity_matrix):
    """Create a readable similarity table.

    Parameters:
        labels: Row and column labels for the compared texts.
        similarity_matrix: Pairwise similarity scores.

    Returns:
        A pandas DataFrame with rounded similarity values.
    """
    # DataFrame giúp hiển thị ma trận tương tự rõ ràng trong notebook.
    return pd.DataFrame(
        np.round(similarity_matrix, 3),
        index=labels,
        columns=labels,
    )


sentences = [
    "A student is learning machine learning.",
    "A learner studies artificial intelligence.",
    "The football team won the match.",
    "The weather is sunny today.",
    "Deep learning models can understand text.",
]

sentence_embeddings = embed_texts(sentences, model)
sentence_similarity = cosine_similarity_matrix(sentence_embeddings)

make_similarity_table([f"S{i+1}" for i in range(len(sentences))], sentence_similarity)

,S1,S2,S3,S4,S5
S1,1.000,0.550,-0.017,0.056,0.274
S2,0.550,1.000,0.014,-0.087,0.327
S3,-0.017,0.014,1.000,0.046,0.060
S4,0.056,-0.087,0.046,1.000,-0.008
S5,0.274,0.327,0.060,-0.008,1.000


In [5]:
def top_sentence_pairs(texts, similarity_matrix, top_k=5):
    """Find the most similar sentence pairs.

    Parameters:
        texts: A list of original sentences.
        similarity_matrix: Pairwise similarity scores for the sentences.
        top_k: Number of top pairs to return.

    Returns:
        A pandas DataFrame containing the most similar sentence pairs.
    """
    pairs = []

    # Chỉ xét nửa trên của ma trận để tránh lặp cặp (i, j) và (j, i).
    for i in range(len(texts)):
        for j in range(i + 1, len(texts)):
            pairs.append(
                {
                    "Sentence A": texts[i],
                    "Sentence B": texts[j],
                    "Similarity": similarity_matrix[i, j],
                }
            )

    # Sắp xếp giảm dần để các cặp gần nghĩa nhất đứng đầu.
    pairs = sorted(pairs, key=lambda item: item["Similarity"], reverse=True)
    return pd.DataFrame(pairs[:top_k])


top_sentence_pairs(sentences, sentence_similarity, top_k=5)

,Sentence A,Sentence B,Similarity
0,A student is learning machine learning.,A learner studies artificial intelligence.,0.550184
1,A learner studies artificial intelligence.,Deep learning models can understand text.,0.327319
2,A student is learning machine learning.,Deep learning models can understand text.,0.273734
3,The football team won the match.,Deep learning models can understand text.,0.059593
4,A student is learning machine learning.,The weather is sunny today.,0.056239


## 3. Compare Two Documents

For a simple classroom demo, we represent a document by:

1. Splitting the document into sentences.
2. Encoding each sentence.
3. Averaging the sentence embeddings.
4. Normalizing the final document vector.

This is simple, fast, and useful as a baseline.

In [6]:
def split_into_sentences(text):
    """Split a document into simple sentence-like units.

    Parameters:
        text: A document string.

    Returns:
        A list of sentence strings.
    """
    # Tách câu đơn giản bằng regex dựa trên dấu chấm, chấm hỏi, và chấm than.
    pieces = re.split(r"(?<=[.!?])\s+", text.strip())
    return [piece for piece in pieces if piece]


def document_embedding(document, embedding_model):
    """Create one embedding vector for a whole document.

    Parameters:
        document: A document string to encode.
        embedding_model: A SentenceTransformer model used to create embeddings.

    Returns:
        A normalized NumPy vector representing the document.
    """
    sentences_in_document = split_into_sentences(document)

    # Mã hóa từng câu, sau đó lấy trung bình để có vector đại diện cho toàn tài liệu.
    sentence_vectors = embed_texts(sentences_in_document, embedding_model)
    doc_vector = sentence_vectors.mean(axis=0)

    # Chuẩn hóa vector tài liệu để có thể dùng tích vô hướng làm cosine similarity.
    norm = np.linalg.norm(doc_vector)
    if norm == 0:
        return doc_vector
    return doc_vector / norm


documents = {
    "AI document": (
        "Artificial intelligence studies how machines can act intelligently. "
        "Natural language processing helps computers understand and generate human language."
    ),
    "Machine learning document": (
        "Machine learning builds models from data. "
        "Deep learning is often used for text classification, translation, and question answering."
    ),
    "Sports document": (
        "The football team trained every morning before the championship. "
        "Fans celebrated after the final match ended."
    ),
}

# Tạo vector đại diện cho từng tài liệu để so sánh ý nghĩa tổng quát giữa các tài liệu.
document_vectors = np.vstack([
    document_embedding(text, model)
    for text in documents.values()
])

document_similarity = cosine_similarity_matrix(document_vectors)
make_similarity_table(list(documents.keys()), document_similarity)

,AI document,Machine learning document,Sports document
AI document,1.000,0.502,0.074
Machine learning document,0.502,1.000,0.051
Sports document,0.074,0.051,1.000


In [7]:
def compare_two_documents(doc_a, doc_b, embedding_model):
    """Compare two documents using their embedding vectors.

    Parameters:
        doc_a: The first document string.
        doc_b: The second document string.
        embedding_model: A SentenceTransformer model used to create embeddings.

    Returns:
        A cosine similarity score between the two documents.
    """
    # Biến mỗi tài liệu thành một vector rồi lấy tích vô hướng giữa hai vector đã chuẩn hóa.
    vector_a = document_embedding(doc_a, embedding_model)
    vector_b = document_embedding(doc_b, embedding_model)
    return float(vector_a @ vector_b)


doc_1 = documents["AI document"]
doc_2 = documents["Machine learning document"]
doc_3 = documents["Sports document"]

print("AI vs. Machine Learning:", round(compare_two_documents(doc_1, doc_2, model), 3))
print("AI vs. Sports:", round(compare_two_documents(doc_1, doc_3, model), 3))

AI vs. Machine Learning: 0.502
AI vs. Sports: 0.074


## 4. A Simple Retrieval-Based Chatbot

This chatbot is trained with a small list of `(question, answer)` pairs.

It does not train neural network weights. Instead, it builds an embedding index:

1. Convert all training questions into vectors.
2. Convert the user question into a vector.
3. Find the closest stored question.
4. Return the answer paired with that question.

In [8]:
qa_pairs = [
    {
        "question": "What is artificial intelligence?",
        "answer": "Artificial intelligence is the study of agents that perceive, reason, and act intelligently.",
    },
    {
        "question": "What is a sentence embedding?",
        "answer": "A sentence embedding is a vector representation that captures the meaning of a sentence.",
    },
    {
        "question": "How can I compare two sentences?",
        "answer": "Encode both sentences as vectors and compute their cosine similarity.",
    },
    {
        "question": "How can I compare two documents?",
        "answer": "Represent each document with an embedding, then compute cosine similarity between the document vectors.",
    },
    {
        "question": "What is natural language processing?",
        "answer": "Natural language processing is the area of AI that studies how computers process human language.",
    },
    {
        "question": "What is a transformer model?",
        "answer": "A transformer is a neural architecture based on attention mechanisms for processing sequences.",
    },
    {
        "question": "What is semantic similarity?",
        "answer": "Semantic similarity measures how close two texts are in meaning, not just how many words they share.",
    },
]

In [ ]:
def build_chatbot_index(training_pairs, embedding_model):
    """Build an embedding index for chatbot training questions.

    Parameters:
        training_pairs: A list of dictionaries with 'question' and 'answer' keys.
        embedding_model: A SentenceTransformer model used to encode questions.

    Returns:
        A NumPy array containing embeddings for all training questions.
    """
    # Lấy toàn bộ câu hỏi mẫu để tạo chỉ mục tìm kiếm theo ngữ nghĩa.
    questions = [pair["question"] for pair in training_pairs]
    question_embeddings = embed_texts(questions, embedding_model)
    return question_embeddings


def answer_question(user_question, training_pairs, question_embeddings, embedding_model, threshold=0.45, top_k=3):
    """Answer a user question by retrieving the nearest training question.

    Parameters:
        user_question: The question entered by the user.
        training_pairs: A list of dictionaries with 'question' and 'answer' keys.
        question_embeddings: Precomputed embeddings for the training questions.
        embedding_model: A SentenceTransformer model used to encode the user question.
        threshold: Minimum similarity score required to return a confident answer.
        top_k: Number of nearest training questions to report for inspection.

    Returns:
        A dictionary containing the answer, matched question, score, and top matches.
    """
    # Biến câu hỏi người dùng thành vector trong cùng không gian với câu hỏi huấn luyện.
    user_embedding = embed_texts([user_question], embedding_model)[0]

    # So sánh câu hỏi người dùng với tất cả câu hỏi đã lưu.
    scores = question_embeddings @ user_embedding
    ranked_indices = np.argsort(scores)[::-1]
    best_index = int(ranked_indices[0])
    best_score = float(scores[best_index])

    # Lưu một vài kết quả gần nhất để người học kiểm tra cách chatbot chọn câu trả lời.
    top_matches = [
        {
            "question": training_pairs[int(index)]["question"],
            "answer": training_pairs[int(index)]["answer"],
            "similarity": float(scores[int(index)]),
        }
        for index in ranked_indices[:top_k]
    ]

    if best_score < threshold:
        answer = "I am not confident enough to answer this question from the training data."
    else:
        answer = training_pairs[best_index]["answer"]

    return {
        "user_question": user_question,
        "answer": answer,
        "matched_question": training_pairs[best_index]["question"],
        "similarity": best_score,
        "top_matches": top_matches,
    }


# Đây là bước 'huấn luyện' theo nghĩa retrieval: tạo vector cho các câu hỏi mẫu.
question_embeddings = build_chatbot_index(qa_pairs, model)

print("Number of training questions:", len(qa_pairs))
print("Question embedding matrix shape:", question_embeddings.shape)

Number of training questions: 7
Question embedding matrix shape: (7, 384)


In [10]:
demo_questions = [
    "How do I turn a sentence into numbers?",
    "How can I measure whether two paragraphs are similar?",
    "Can you explain transformer models?",
    "What is the cafeteria menu today?",
]

for question in demo_questions:
    result = answer_question(question, qa_pairs, question_embeddings, model)

    print("=" * 90)
    print("User question:", result["user_question"])
    print("Best matched training question:", result["matched_question"])
    print("Similarity:", round(result["similarity"], 3))
    print("Answer:", result["answer"])

User question: How do I turn a sentence into numbers?
Best matched training question: How can I compare two sentences?
Similarity: 0.405
Answer: I am not confident enough to answer this question from the training data.
User question: How can I measure whether two paragraphs are similar?
Best matched training question: How can I compare two documents?
Similarity: 0.636
Answer: Represent each document with an embedding, then compute cosine similarity between the document vectors.
User question: Can you explain transformer models?
Best matched training question: What is a transformer model?
Similarity: 0.925
Answer: A transformer is a neural architecture based on attention mechanisms for processing sequences.
User question: What is the cafeteria menu today?
Best matched training question: What is a transformer model?
Similarity: 0.137
Answer: I am not confident enough to answer this question from the training data.


### Inspect the Nearest Questions

The next cell shows the top retrieved training questions for one user question. This helps explain why the chatbot selected its answer.

In [11]:
user_question = "How do embeddings help compare text meaning?"
result = answer_question(user_question, qa_pairs, question_embeddings, model, top_k=3)

pd.DataFrame(result["top_matches"])

,question,answer,similarity
0,What is a sentence embedding?,A sentence embedding is a vector representatio...,0.665576
1,What is semantic similarity?,Semantic similarity measures how close two tex...,0.518912
2,How can I compare two sentences?,Encode both sentences as vectors and compute t...,0.486137


## 5. Summary

In this notebook, we used `SentenceTransformer` to:

- Encode sentences as numerical vectors.
- Compare sentence meanings with cosine similarity.
- Build simple document vectors by averaging sentence vectors.
- Create a retrieval chatbot that answers by finding the nearest training question.

This retrieval approach is simple, interpretable, and useful for small knowledge bases. For larger systems, the same idea can be extended with vector databases, better chunking, and stronger embedding models.